In [1]:

"""
CN7 05 - PR-AUC / Recall / Precision 운영점 분석
================================================

목적
----
CN7_04에서 생성한 OOF prediction을 이용해
"모델의 순위 성능"과 "실제 사용할 threshold"를 분리해서 분석한다.

핵심 지표
---------
1) PR-AUC
2) Recall >= 90/80/70/60/50% 일 때 가능한 최고 Precision
3) Precision >= 40/30/20/10% 일 때 가능한 최고 Recall
4) Top 1/2/5/10% 검사 시 Recall / Precision
5) threshold별 TP / FP / FN / TN
6) PR curve 및 운영점 시각화

주의
----
- ROC-AUC는 참고용으로만 저장한다.
- threshold는 OOF prediction에서 탐색한다.
- OOF에서 고른 threshold 성능은 "운영 후보"이지,
  완전히 독립된 최종 성능 추정치로 보지 않는다.
- 반복 Group CV와 Purged Time-Block CV를 반드시 따로 본다.
"""

from pathlib import Path
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# =============================================================================
# 0. 경로 / 설정
# =============================================================================

CN7_WORK_DIR = Path(
    r"C:\Users\Administrator\Desktop\kamp\다시하는 cn7"
)

VALID_DIR = CN7_WORK_DIR / "cn7_04_validation"

GROUP_OOF_PATH = VALID_DIR / "repeated_group_oof.csv"
BLOCK_OOF_PATH = VALID_DIR / "purged_block_oof.csv"

OUT_DIR = CN7_WORK_DIR / "cn7_05_pr_analysis"
PLOT_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "target"
PRED_COL = "prediction"

RECALL_TARGETS = [0.90, 0.80, 0.70, 0.60, 0.50]
PRECISION_TARGETS = [0.40, 0.30, 0.20, 0.10]
TOP_FRACS = [0.01, 0.02, 0.05, 0.10]

# threshold 전체표를 너무 크게 만들고 싶지 않으면 True
SAVE_ALL_THRESHOLDS = True


# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print("\n" + "=" * 115)
    print(title)
    print("=" * 115)


def safe_metric(v):
    if v is None:
        return np.nan
    try:
        if np.isfinite(v):
            return float(v)
    except Exception:
        pass
    return np.nan


def confusion_at_threshold(y_true, score, threshold):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    pred = (score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        pred,
        labels=[0, 1],
    ).ravel()

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )
    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    predicted_positive_rate = (tp + fp) / len(y_true)

    return {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "predicted_positive": int(tp + fp),
        "predicted_positive_rate": float(predicted_positive_rate),
    }


def make_threshold_table(y_true, score):
    """
    score의 모든 고유값을 threshold 후보로 사용.
    이 방식은 positive set이 바뀌는 모든 지점을 정확히 확인한다.
    """
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    valid = np.isfinite(score)
    y = y_true[valid]
    s = score[valid]

    thresholds = np.sort(
        np.unique(s)
    )[::-1]

    # 아무 것도 positive가 되지 않는 threshold도 한 개 추가
    thresholds = np.r_[
        np.nextafter(thresholds[0], np.inf),
        thresholds
    ]

    rows = [
        confusion_at_threshold(
            y,
            s,
            threshold=t,
        )
        for t in thresholds
    ]

    df = pd.DataFrame(rows)

    return df


def best_precision_at_recall(threshold_df, recall_target):
    """
    Recall >= target 조건을 만족하는 점 중 precision 최대.
    tie-break:
      1) precision 높은 순
      2) recall 높은 순
      3) predicted_positive 적은 순
    """
    cand = threshold_df[
        threshold_df["recall"] >= recall_target
    ].copy()

    if len(cand) == 0:
        return None

    cand = cand.sort_values(
        [
            "precision",
            "recall",
            "predicted_positive",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )

    return cand.iloc[0]


def best_recall_at_precision(threshold_df, precision_target):
    """
    Precision >= target 조건을 만족하는 점 중 recall 최대.
    tie-break:
      1) recall 높은 순
      2) precision 높은 순
      3) predicted_positive 적은 순
    """
    cand = threshold_df[
        threshold_df["precision"] >= precision_target
    ].copy()

    if len(cand) == 0:
        return None

    cand = cand.sort_values(
        [
            "recall",
            "precision",
            "predicted_positive",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )

    return cand.iloc[0]


def best_f1(threshold_df):
    cand = threshold_df.sort_values(
        [
            "f1",
            "recall",
            "precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    return cand.iloc[0]


def top_fraction_metrics(y_true, score, frac):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    valid = np.isfinite(score)
    y = y_true[valid]
    s = score[valid]

    n = len(y)
    k = max(
        1,
        int(np.ceil(n * frac)),
    )

    order = np.argsort(-s)
    top = order[:k]

    tp = int(y[top].sum())
    fp = int(k - tp)
    total_pos = int(y.sum())
    fn = int(total_pos - tp)

    precision = tp / k
    recall = (
        tp / total_pos
        if total_pos > 0
        else np.nan
    )

    threshold = float(
        s[order[k - 1]]
    )

    return {
        "fraction": frac,
        "k": k,
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
    }


def pr_auc_and_roc(y_true, score):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    valid = np.isfinite(score)
    y = y_true[valid]
    s = score[valid]

    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan, np.nan

    return (
        float(
            average_precision_score(
                y,
                s,
            )
        ),
        float(
            roc_auc_score(
                y,
                s,
            )
        ),
    )


def add_meta(row, meta):
    out = dict(meta)
    out.update(row)
    return out


# =============================================================================
# 1. OOF 로드
# =============================================================================

banner("CN7 05 - PR-AUC / Recall / Precision 운영점 분석")

if not GROUP_OOF_PATH.exists():
    raise FileNotFoundError(
        f"없음: {GROUP_OOF_PATH}"
    )

if not BLOCK_OOF_PATH.exists():
    raise FileNotFoundError(
        f"없음: {BLOCK_OOF_PATH}"
    )

group_oof = pd.read_csv(GROUP_OOF_PATH)
block_oof = pd.read_csv(BLOCK_OOF_PATH)

print("group_oof:", group_oof.shape)
print("block_oof:", block_oof.shape)

datasets = {
    "repeated_stratified_group": group_oof,
    "purged_time_block": block_oof,
}


# =============================================================================
# 2. 기본 검증
# =============================================================================

banner("1) OOF 기본 검증")

for cv_name, d in datasets.items():
    print(f"\n[{cv_name}]")

    required = {
        "feature_set",
        "model",
        TARGET_COL,
        PRED_COL,
    }

    missing = required - set(d.columns)

    if missing:
        raise KeyError(
            f"{cv_name} 누락 컬럼: {missing}"
        )

    print(
        d.groupby(
            ["feature_set", "model"]
        )
        .agg(
            rows=(TARGET_COL, "size"),
            defects=(TARGET_COL, "sum"),
            pred_na=(PRED_COL, lambda x: x.isna().sum()),
        )
        .to_string()
    )


# =============================================================================
# 3. 모델별 PR / threshold 분석
# =============================================================================

banner("2) 모델별 PR 운영점")

global_summary = []
recall_constraint_rows = []
precision_constraint_rows = []
top_fraction_rows = []
best_f1_rows = []
threshold_all = []

for cv_name, d in datasets.items():

    for (
        feature_set,
        model_name
    ), part in d.groupby(
        ["feature_set", "model"]
    ):
        part = part.sort_values(
            "row_index"
        ).copy()

        valid = part[PRED_COL].notna()

        y = (
            part.loc[
                valid,
                TARGET_COL,
            ]
            .astype(int)
            .to_numpy()
        )

        score = (
            part.loc[
                valid,
                PRED_COL,
            ]
            .astype(float)
            .to_numpy()
        )

        if len(y) == 0:
            continue

        pr_auc, roc_auc = pr_auc_and_roc(
            y,
            score,
        )

        threshold_df = make_threshold_table(
            y,
            score,
        )

        meta = {
            "cv": cv_name,
            "feature_set": feature_set,
            "model": model_name,
            "n": len(y),
            "defects": int(y.sum()),
            "defect_rate": float(y.mean()),
        }

        # 전체 summary
        global_summary.append({
            **meta,
            "PR_AUC": pr_auc,
            "ROC_AUC_reference": roc_auc,
        })

        # Recall 제약
        for target_recall in RECALL_TARGETS:
            r = best_precision_at_recall(
                threshold_df,
                target_recall,
            )

            row = {
                **meta,
                "target_recall": target_recall,
            }

            if r is None:
                row.update({
                    "threshold": np.nan,
                    "precision": np.nan,
                    "recall": np.nan,
                    "TP": np.nan,
                    "FP": np.nan,
                    "FN": np.nan,
                    "predicted_positive": np.nan,
                    "predicted_positive_rate": np.nan,
                })
            else:
                row.update({
                    "threshold": float(r["threshold"]),
                    "precision": float(r["precision"]),
                    "recall": float(r["recall"]),
                    "TP": int(r["TP"]),
                    "FP": int(r["FP"]),
                    "FN": int(r["FN"]),
                    "predicted_positive": int(
                        r["predicted_positive"]
                    ),
                    "predicted_positive_rate": float(
                        r["predicted_positive_rate"]
                    ),
                })

            recall_constraint_rows.append(row)

        # Precision 제약
        for target_precision in PRECISION_TARGETS:
            r = best_recall_at_precision(
                threshold_df,
                target_precision,
            )

            row = {
                **meta,
                "target_precision": target_precision,
            }

            if r is None:
                row.update({
                    "threshold": np.nan,
                    "precision": np.nan,
                    "recall": np.nan,
                    "TP": np.nan,
                    "FP": np.nan,
                    "FN": np.nan,
                    "predicted_positive": np.nan,
                    "predicted_positive_rate": np.nan,
                })
            else:
                row.update({
                    "threshold": float(r["threshold"]),
                    "precision": float(r["precision"]),
                    "recall": float(r["recall"]),
                    "TP": int(r["TP"]),
                    "FP": int(r["FP"]),
                    "FN": int(r["FN"]),
                    "predicted_positive": int(
                        r["predicted_positive"]
                    ),
                    "predicted_positive_rate": float(
                        r["predicted_positive_rate"]
                    ),
                })

            precision_constraint_rows.append(row)

        # Best F1
        bf = best_f1(threshold_df)

        best_f1_rows.append({
            **meta,
            "threshold": float(bf["threshold"]),
            "precision": float(bf["precision"]),
            "recall": float(bf["recall"]),
            "f1": float(bf["f1"]),
            "TP": int(bf["TP"]),
            "FP": int(bf["FP"]),
            "FN": int(bf["FN"]),
            "TN": int(bf["TN"]),
            "predicted_positive": int(
                bf["predicted_positive"]
            ),
            "predicted_positive_rate": float(
                bf["predicted_positive_rate"]
            ),
        })

        # Top fraction
        for frac in TOP_FRACS:
            tm = top_fraction_metrics(
                y,
                score,
                frac,
            )

            top_fraction_rows.append({
                **meta,
                **tm,
            })

        # 모든 threshold 저장
        if SAVE_ALL_THRESHOLDS:
            t = threshold_df.copy()

            for k, v in meta.items():
                t[k] = v

            threshold_all.append(t)

        print(
            f"{cv_name:28s} | "
            f"{feature_set:20s} | "
            f"{model_name:10s} | "
            f"PR-AUC={pr_auc:.4f}"
        )

# =============================================================================
# 4. 결과 DataFrame
# =============================================================================

summary_df = pd.DataFrame(
    global_summary
).sort_values(
    ["cv", "PR_AUC"],
    ascending=[True, False],
)

recall_df = pd.DataFrame(
    recall_constraint_rows
).sort_values(
    [
        "cv",
        "target_recall",
        "precision",
    ],
    ascending=[
        True,
        False,
        False,
    ],
)

precision_df = pd.DataFrame(
    precision_constraint_rows
).sort_values(
    [
        "cv",
        "target_precision",
        "recall",
    ],
    ascending=[
        True,
        False,
        False,
    ],
)

top_df = pd.DataFrame(
    top_fraction_rows
).sort_values(
    [
        "cv",
        "fraction",
        "recall",
        "precision",
    ],
    ascending=[
        True,
        True,
        False,
        False,
    ],
)

f1_df = pd.DataFrame(
    best_f1_rows
).sort_values(
    [
        "cv",
        "f1",
    ],
    ascending=[
        True,
        False,
    ],
)

summary_df.to_csv(
    OUT_DIR / "pr_auc_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

recall_df.to_csv(
    OUT_DIR / "best_precision_at_recall.csv",
    index=False,
    encoding="utf-8-sig",
)

precision_df.to_csv(
    OUT_DIR / "best_recall_at_precision.csv",
    index=False,
    encoding="utf-8-sig",
)

top_df.to_csv(
    OUT_DIR / "top_fraction_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

f1_df.to_csv(
    OUT_DIR / "best_f1_threshold.csv",
    index=False,
    encoding="utf-8-sig",
)

if threshold_all:
    threshold_all_df = pd.concat(
        threshold_all,
        ignore_index=True,
    )

    threshold_all_df.to_csv(
        OUT_DIR / "all_threshold_metrics.csv",
        index=False,
        encoding="utf-8-sig",
    )


# =============================================================================
# 5. PR-AUC
# =============================================================================

banner("3) PR-AUC")

print(
    summary_df[
        [
            "cv",
            "feature_set",
            "model",
            "n",
            "defects",
            "defect_rate",
            "PR_AUC",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# 6. Recall 목표별 최고 Precision
# =============================================================================

banner("4) Recall 목표별 가능한 최고 Precision")

display_cols = [
    "cv",
    "feature_set",
    "model",
    "target_recall",
    "threshold",
    "recall",
    "precision",
    "TP",
    "FP",
    "FN",
    "predicted_positive",
    "predicted_positive_rate",
]

print(
    recall_df[
        display_cols
    ]
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# 7. Precision 목표별 최고 Recall
# =============================================================================

banner("5) Precision 목표별 가능한 최고 Recall")

display_cols2 = [
    "cv",
    "feature_set",
    "model",
    "target_precision",
    "threshold",
    "precision",
    "recall",
    "TP",
    "FP",
    "FN",
    "predicted_positive",
    "predicted_positive_rate",
]

print(
    precision_df[
        display_cols2
    ]
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# 8. Top fraction
# =============================================================================

banner("6) 상위 위험군 비율별 성능")

print(
    top_df[
        [
            "cv",
            "feature_set",
            "model",
            "fraction",
            "k",
            "TP",
            "FP",
            "FN",
            "precision",
            "recall",
            "threshold",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# 9. Best F1 - 참고
# =============================================================================

banner("7) Best F1 threshold - 참고")

print(
    f1_df[
        [
            "cv",
            "feature_set",
            "model",
            "threshold",
            "precision",
            "recall",
            "f1",
            "TP",
            "FP",
            "FN",
            "predicted_positive",
        ]
    ]
    .round(4)
    .to_string(index=False)
)


# =============================================================================
# 10. PR Curve 저장
# =============================================================================

banner("8) PR Curve 저장")

for cv_name, d in datasets.items():

    fig = plt.figure(
        figsize=(8, 6)
    )

    for (
        feature_set,
        model_name
    ), part in d.groupby(
        ["feature_set", "model"]
    ):
        part = part.sort_values(
            "row_index"
        )

        valid = part[PRED_COL].notna()

        y = (
            part.loc[
                valid,
                TARGET_COL,
            ]
            .astype(int)
            .to_numpy()
        )

        score = (
            part.loc[
                valid,
                PRED_COL,
            ]
            .astype(float)
            .to_numpy()
        )

        if len(y) == 0:
            continue

        precision, recall, thresholds = (
            precision_recall_curve(
                y,
                score,
            )
        )

        ap = average_precision_score(
            y,
            score,
        )

        plt.plot(
            recall,
            precision,
            label=(
                f"{feature_set} / "
                f"{model_name} "
                f"(AP={ap:.3f})"
            ),
        )

    prevalence = (
        d[TARGET_COL].astype(int).mean()
    )

    plt.axhline(
        prevalence,
        linestyle="--",
        linewidth=1,
        label=(
            f"Random baseline="
            f"{prevalence:.3f}"
        ),
    )

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(
        f"CN7 Precision-Recall Curve\n{cv_name}"
    )
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.legend()
    plt.tight_layout()

    path = (
        PLOT_DIR
        / f"pr_curve__{cv_name}.png"
    )

    plt.savefig(
        path,
        dpi=160,
    )
    plt.close(fig)

    print(path)


# =============================================================================
# 11. Recall 목표별 Precision 그래프
# =============================================================================

banner("9) 운영점 그래프 저장")

for cv_name in recall_df["cv"].unique():

    part = recall_df[
        recall_df["cv"] == cv_name
    ].copy()

    fig = plt.figure(
        figsize=(9, 6)
    )

    for (
        feature_set,
        model_name
    ), g in part.groupby(
        ["feature_set", "model"]
    ):
        g = g.sort_values(
            "target_recall"
        )

        plt.plot(
            g["target_recall"],
            g["precision"],
            marker="o",
            label=(
                f"{feature_set} / "
                f"{model_name}"
            ),
        )

    plt.xlabel("Required Recall")
    plt.ylabel("Best Available Precision")
    plt.title(
        f"Precision at Recall Constraint\n{cv_name}"
    )
    plt.xlim(0.45, 0.95)
    plt.ylim(
        0,
        max(
            0.5,
            part["precision"].max() * 1.1
        ),
    )
    plt.legend()
    plt.tight_layout()

    path = (
        PLOT_DIR
        / f"precision_at_recall__{cv_name}.png"
    )

    plt.savefig(
        path,
        dpi=160,
    )
    plt.close(fig)

    print(path)


# =============================================================================
# 12. Operational shortlist
# =============================================================================

banner("10) 운영 후보 Shortlist")

shortlist_rows = []

for cv_name in summary_df["cv"].unique():

    # 이 CV에서 PR-AUC 상위 모델
    cv_sum = summary_df[
        summary_df["cv"] == cv_name
    ].copy()

    best_pr = cv_sum.sort_values(
        "PR_AUC",
        ascending=False,
    ).iloc[0]

    shortlist_rows.append({
        "cv": cv_name,
        "criterion": "highest_PR_AUC",
        "feature_set": best_pr["feature_set"],
        "model": best_pr["model"],
        "PR_AUC": best_pr["PR_AUC"],
        "constraint": "",
        "threshold": np.nan,
        "precision": np.nan,
        "recall": np.nan,
    })

    # Recall >= 0.8에서 precision 최고
    cand = recall_df[
        (recall_df["cv"] == cv_name)
        &
        (np.isclose(
            recall_df["target_recall"],
            0.80,
        ))
    ].copy()

    if len(cand):
        cand = cand.sort_values(
            [
                "precision",
                "recall",
            ],
            ascending=[
                False,
                False,
            ],
        )
        r = cand.iloc[0]

        shortlist_rows.append({
            "cv": cv_name,
            "criterion": "best_precision_at_recall>=0.80",
            "feature_set": r["feature_set"],
            "model": r["model"],
            "PR_AUC": float(
                cv_sum[
                    (cv_sum["feature_set"] == r["feature_set"])
                    &
                    (cv_sum["model"] == r["model"])
                ]["PR_AUC"].iloc[0]
            ),
            "constraint": "recall>=0.80",
            "threshold": r["threshold"],
            "precision": r["precision"],
            "recall": r["recall"],
        })

    # Recall >= 0.9에서 precision 최고
    cand = recall_df[
        (recall_df["cv"] == cv_name)
        &
        (np.isclose(
            recall_df["target_recall"],
            0.90,
        ))
    ].copy()

    if len(cand):
        cand = cand.sort_values(
            [
                "precision",
                "recall",
            ],
            ascending=[
                False,
                False,
            ],
        )
        r = cand.iloc[0]

        shortlist_rows.append({
            "cv": cv_name,
            "criterion": "best_precision_at_recall>=0.90",
            "feature_set": r["feature_set"],
            "model": r["model"],
            "PR_AUC": float(
                cv_sum[
                    (cv_sum["feature_set"] == r["feature_set"])
                    &
                    (cv_sum["model"] == r["model"])
                ]["PR_AUC"].iloc[0]
            ),
            "constraint": "recall>=0.90",
            "threshold": r["threshold"],
            "precision": r["precision"],
            "recall": r["recall"],
        })

    # precision >= .20에서 recall 최고
    cand = precision_df[
        (precision_df["cv"] == cv_name)
        &
        (np.isclose(
            precision_df["target_precision"],
            0.20,
        ))
    ].copy()

    if len(cand):
        cand = cand.sort_values(
            [
                "recall",
                "precision",
            ],
            ascending=[
                False,
                False,
            ],
        )
        r = cand.iloc[0]

        shortlist_rows.append({
            "cv": cv_name,
            "criterion": "best_recall_at_precision>=0.20",
            "feature_set": r["feature_set"],
            "model": r["model"],
            "PR_AUC": float(
                cv_sum[
                    (cv_sum["feature_set"] == r["feature_set"])
                    &
                    (cv_sum["model"] == r["model"])
                ]["PR_AUC"].iloc[0]
            ),
            "constraint": "precision>=0.20",
            "threshold": r["threshold"],
            "precision": r["precision"],
            "recall": r["recall"],
        })

shortlist_df = pd.DataFrame(
    shortlist_rows
)

print(
    shortlist_df
    .round(4)
    .to_string(index=False)
)

shortlist_df.to_csv(
    OUT_DIR / "operational_shortlist.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 13. 자동 요약
# =============================================================================

banner("11) 자동 요약")

lines = []

for _, r in summary_df.iterrows():
    lines.append(
        f"- {r['cv']} | "
        f"{r['feature_set']} | "
        f"{r['model']} | "
        f"PR-AUC={r['PR_AUC']:.4f}"
    )

# Group CV만 별도 운영점
group_recall80 = recall_df[
    (recall_df["cv"] == "repeated_stratified_group")
    &
    (np.isclose(
        recall_df["target_recall"],
        0.80,
    ))
].copy()

if len(group_recall80):
    r = group_recall80.sort_values(
        [
            "precision",
            "recall",
        ],
        ascending=[
            False,
            False,
        ],
    ).iloc[0]

    lines.append(
        f"- Group CV / Recall>=80% 최고 Precision: "
        f"{r['feature_set']} + {r['model']} | "
        f"Recall={r['recall']:.3f}, "
        f"Precision={r['precision']:.3f}, "
        f"threshold={r['threshold']:.6f}, "
        f"TP={int(r['TP'])}, FP={int(r['FP'])}"
    )

group_recall90 = recall_df[
    (recall_df["cv"] == "repeated_stratified_group")
    &
    (np.isclose(
        recall_df["target_recall"],
        0.90,
    ))
].copy()

if len(group_recall90):
    r = group_recall90.sort_values(
        [
            "precision",
            "recall",
        ],
        ascending=[
            False,
            False,
        ],
    ).iloc[0]

    lines.append(
        f"- Group CV / Recall>=90% 최고 Precision: "
        f"{r['feature_set']} + {r['model']} | "
        f"Recall={r['recall']:.3f}, "
        f"Precision={r['precision']:.3f}, "
        f"threshold={r['threshold']:.6f}, "
        f"TP={int(r['TP'])}, FP={int(r['FP'])}"
    )

block_recall80 = recall_df[
    (recall_df["cv"] == "purged_time_block")
    &
    (np.isclose(
        recall_df["target_recall"],
        0.80,
    ))
].copy()

if len(block_recall80):
    r = block_recall80.sort_values(
        [
            "precision",
            "recall",
        ],
        ascending=[
            False,
            False,
        ],
    ).iloc[0]

    lines.append(
        f"- Purged CV / Recall>=80% 최고 Precision: "
        f"{r['feature_set']} + {r['model']} | "
        f"Recall={r['recall']:.3f}, "
        f"Precision={r['precision']:.3f}, "
        f"threshold={r['threshold']:.6f}, "
        f"TP={int(r['TP'])}, FP={int(r['FP'])}"
    )

lines.append(
    "- 주의: OOF에서 선택한 threshold는 운영 후보이며, "
    "완전히 독립된 최종 성능 추정치는 아니다."
)

for line in lines:
    print(line)

with open(
    OUT_DIR / "pr_operating_summary.txt",
    "w",
    encoding="utf-8",
) as f:
    f.write("\n".join(lines))


# =============================================================================
# 완료
# =============================================================================

banner("완료")

print("결과 폴더:", OUT_DIR)
print()
print("핵심 파일:")
print("  1) pr_auc_summary.csv")
print("  2) best_precision_at_recall.csv")
print("  3) best_recall_at_precision.csv")
print("  4) top_fraction_metrics.csv")
print("  5) best_f1_threshold.csv")
print("  6) operational_shortlist.csv")
print("  7) all_threshold_metrics.csv")
print("  8) pr_operating_summary.txt")
print("  9) plots/")
print()
print(
    "다음 단계는 이 결과를 바탕으로 "
    "PR-AUC/Recall/Precision을 실제로 올리는 feature engineering입니다."
)



CN7 05 - PR-AUC / Recall / Precision 운영점 분석
group_oof: (4844, 10)
block_oof: (4844, 11)

1) OOF 기본 검증

[repeated_stratified_group]
                              rows  defects  pred_na
feature_set       model                             
sensors_only      extratrees  1211       17        0
                  logreg      1211       17        0
sensors_plus_side extratrees  1211       17        0
                  logreg      1211       17        0

[purged_time_block]
                              rows  defects  pred_na
feature_set       model                             
sensors_only      extratrees  1211       17        0
                  logreg      1211       17        0
sensors_plus_side extratrees  1211       17        0
                  logreg      1211       17        0

2) 모델별 PR 운영점
repeated_stratified_group    | sensors_only         | extratrees | PR-AUC=0.4419
repeated_stratified_group    | sensors_only         | logreg     | PR-AUC=0.4787
repeated_stratified_group    | sen